# Практика 3 — Преобразование и JOIN — профиль организаций HMDA

**ФИО и группа:** Кулик Георгий, Т26ОАД-Т.МО31.

## Подготовка

In [1]:
from pathlib import Path
import json
import pandas as pd
from pandas.testing import assert_frame_equal

DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)

decisions = pd.read_csv(
    DATA / "decisions.csv",
    dtype={"lei": "string", "county_code": "string"},
)

## 01. Входы, строка и ключ

In [2]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei":"string"})
assert len(organizations) == 470
assert not organizations.duplicated(org_keys).any()
assert not organizations[org_keys].isna().any().any()
assert not decisions[org_keys].isna().any().any()
organizations.head(4)
assert decisions["action_taken"].isin([1, 2, 3]).all()
display(decisions.head(4))
display(organizations.head(4))

   source_row  activity_year  ... loan_amount income
0           2           2023  ...    275000.0   72.0
1           3           2023  ...    585000.0  174.0
2           5           2023  ...    275000.0  250.0
3           7           2023  ...    295000.0   63.0

[4 rows x 7 columns]

                    lei  ... activity_year
0  549300AG64NHILB7ZP05  ...          2023
1  QFROUN1UWUYU0DVIWD51  ...          2023
2  5493003GQDUH26DNNH17  ...          2023
3  549300DD5QQUHO6PCH70  ...          2023

[4 rows x 4 columns]

## 01. Входы, строка и ключ — шаг 2

In [3]:
long = decisions.groupby(long_keys, dropna=False).size().rename("n").reset_index()
print(long.shape, long["n"].sum())
assert not long.duplicated(long_keys).any()
assert long["n"].sum() == len(decisions)
long.head(6)

(500, 4) 6830


                    lei  activity_year  action_taken   n
0  03D0JEWFDFUS0SEEKG89           2023             1  76
1  03D0JEWFDFUS0SEEKG89           2023             3  15
2  1IE8VN30JCEQV1H4R804           2023             1   5
3  1IE8VN30JCEQV1H4R804           2023             3   2
4  213800KVD76WKI6PN868           2023             1   1
5  213800KVD76WKI6PN868           2023             3   1

## A. Завершите разворот

In [4]:
wide_task = long.pivot(index=org_keys, columns="action_taken", values="n")
print("Размер:", wide_task.shape)
print("Уровни индекса:", wide_task.index.names)
print("Пропуски по исходам:\n", wide_task.isna().sum())
# Пропуск означает, что для этой организации в исходных данных не встретился такой исход.
# pivot требует одну строку на каждую комбинацию индекса и столбца; long это гарантирует.
wide_task.head(4)

Размер: (314, 3)
Уровни индекса: ['lei', 'activity_year']
Пропуски по исходам:
 action_taken
1     20
2    252
3    170
dtype: int64


action_taken                           1    2     3
lei                  activity_year                 
03D0JEWFDFUS0SEEKG89 2023           76.0  NaN  15.0
1IE8VN30JCEQV1H4R804 2023            5.0  NaN   2.0
213800KVD76WKI6PN868 2023            1.0  NaN   1.0
213800QUAI2VH5YM6310 2023           76.0  4.0   9.0

## 02. Обоснованный ноль и явные колонки

In [5]:
wide = long.pivot(index=org_keys, columns="action_taken", values="n")
wide = wide.reindex(columns=[1, 2, 3]).fillna(0).astype("int64")
wide = wide.rename(columns={1:"issued", 2:"approved_not_accepted", 3:"denied"})
wide = wide.rename_axis(None, axis=1).reset_index()
value_cols = ["issued", "approved_not_accepted", "denied"]
assert wide[value_cols].sum().sum() == 6830
wide.head(4)

                    lei  activity_year  issued  approved_not_accepted  denied
0  03D0JEWFDFUS0SEEKG89           2023      76                      0      15
1  1IE8VN30JCEQV1H4R804           2023       5                      0       2
2  213800KVD76WKI6PN868           2023       1                      0       1
3  213800QUAI2VH5YM6310           2023      76                      4       9

## 02. Обоснованный ноль и явные колонки — шаг 2

In [6]:
restored = wide.melt(id_vars=org_keys, value_vars=value_cols, var_name="outcome", value_name="n")
print(restored.shape)
restored.head(6)

(942, 4)


                    lei  activity_year outcome   n
0  03D0JEWFDFUS0SEEKG89           2023  issued  76
1  1IE8VN30JCEQV1H4R804           2023  issued   5
2  213800KVD76WKI6PN868           2023  issued   1
3  213800QUAI2VH5YM6310           2023  issued  76
4  213800XR2TCBQJSF1X93           2023  issued  12
5  254900114YHTRFBALC74           2023  issued   6

## B. Восстановите наблюдаемые комбинации

In [7]:
restored_b = restored.copy()
restored_b["action_taken"] = restored_b["outcome"].map({
    "issued": 1, "approved_not_accepted": 2, "denied": 3
}).astype("int64")
restored_b = restored_b.loc[restored_b["n"].gt(0), long_keys + ["n"]]
restored_b = restored_b.sort_values(long_keys).reset_index(drop=True)
expected = long.sort_values(long_keys).reset_index(drop=True)
assert_frame_equal(restored_b, expected, check_exact=True)
print("Совпало с исходной long-сводкой:", restored_b.shape)
# Нулевые сочетания исключены: их не было в исходной наблюдаемой сводке.
assert restored_b["n"].sum() == len(decisions)
print("Всего сочетаний:", len(restored), "Наблюдаемых:", len(restored_b), "Сумма n:", restored_b["n"].sum())

Совпало с исходной long-сводкой: (500, 4)
Всего сочетаний: 942 Наблюдаемых: 500 Сумма n: 6830


## 03. Добавляем имя с контрактом

In [8]:
lookup = organizations[org_keys + ["organization_name"]].copy()
enriched = decisions.merge(lookup, on=org_keys, how="left", validate="many_to_one", indicator=True)
print(enriched.shape)
print(enriched["_merge"].value_counts())
assert len(enriched) == 6830
assert enriched["source_row"].is_unique
assert set(enriched["source_row"]) == set(decisions["source_row"])
assert enriched["_merge"].eq("both").all()
enriched[["source_row", "lei", "organization_name", "action_taken"]].head(5)

(6830, 9)
_merge
both          6830
left_only        0
right_only       0
Name: count, dtype: int64


   source_row  ... action_taken
0           2  ...            1
1           3  ...            1
2           5  ...            1
3           7  ...            1
4           8  ...            1

[5 rows x 4 columns]

## C. Проверьте сохранность самих значений

In [9]:
original_cols = decisions.columns.tolist()
check = enriched.sort_values("source_row")[original_cols].reset_index(drop=True)
original = decisions.sort_values("source_row")[original_cols].reset_index(drop=True)
assert_frame_equal(check, original, check_exact=True)
print("Исходные значения и порядок строк сохранены.")
# Проверяем, что проверка обнаруживает изменение значения, а не только числа строк.
tampered = enriched.copy()
row = tampered["loan_amount"].first_valid_index()
tampered.loc[row, "loan_amount"] += 1
try:
    assert_frame_equal(
        tampered.sort_values("source_row")[original_cols].reset_index(drop=True),
        original, check_exact=True,
    )
except AssertionError:
    print("Изменение loan_amount корректно обнаружено.")
else:
    raise AssertionError("Проверка не обнаружила изменение значения")
# Одна лишь длина не ловит замену или перестановку значений при неизменном числе строк.

Исходные значения и порядок строк сохранены.
Изменение loan_amount корректно обнаружено.


## 04. Неполная доставка: потеря при inner

In [10]:
partial_lookup = lookup.iloc[1:].copy()
left_partial = decisions.merge(partial_lookup, on=org_keys, how="left", validate="many_to_one", indicator=True)
inner_partial = decisions.merge(partial_lookup, on=org_keys, how="inner", validate="many_to_one")
print(len(left_partial), len(inner_partial))
print(left_partial["_merge"].value_counts())
assert len(left_partial) == 6830
assert len(inner_partial) == 6805

6830 6805
_merge
both          6805
left_only       25
right_only       0
Name: count, dtype: int64


## 04. Неполная доставка: потеря при inner — шаг 2

In [11]:
replayed_lookup = pd.concat([lookup, lookup.iloc[[0]]], ignore_index=True)
wrong_join = decisions.merge(replayed_lookup, on=org_keys, how="left")
print("Без validate:", len(wrong_join))
assert len(wrong_join) == 6855
try:
    decisions.merge(replayed_lookup, on=org_keys, how="left", validate="many_to_one")
except pd.errors.MergeError:
    print("Повторный правый ключ остановлен")
else:
    raise AssertionError("Ожидалась остановка JOIN")

Без validate: 6855
Повторный правый ключ остановлен


## D. Сохраните аудит покрытия

In [12]:
unmatched = left_partial.loc[left_partial["_merge"].eq("left_only"),
                           ["source_row"] + org_keys + ["action_taken"]].sort_values("source_row")
lost = set(decisions["source_row"]) - set(inner_partial["source_row"])
assert lost == set(unmatched["source_row"])
assert len(unmatched) == len(decisions) - len(inner_partial)
assert len(wrong_join) - len(decisions) == len(unmatched)
unmatched.to_csv(WORK / "unmatched.csv", index=False)
audit = {
    "decisions_rows": int(len(decisions)),
    "organizations_rows": int(len(organizations)),
    "partial_lookup_rows": int(len(partial_lookup)),
    "left_join_rows": int(len(left_partial)),
    "left_only_rows": int(left_partial["_merge"].eq("left_only").sum()),
    "inner_join_rows": int(len(inner_partial)),
    "rows_lost_by_inner": int(len(decisions) - len(inner_partial)),
    "duplicate_lookup_join_rows": int(len(wrong_join)),
    "rows_added_by_duplicate_key": int(len(wrong_join) - len(decisions)),
    "expected_validation": "many_to_one",
    "note": "Учебная неполная копия справочника: удалена первая строка lookup.",
}
(WORK / "join_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print(audit)
# LEFT сохраняет все решения и помечает отсутствующие соответствия; INNER отбрасывает их.
# Дубли справа размножают совпавшие строки; validate='many_to_one' ловит нарушение ключа.

{'decisions_rows': 6830, 'organizations_rows': 470, 'partial_lookup_rows': 469, 'left_join_rows': 6830, 'left_only_rows': 25, 'inner_join_rows': 6805, 'rows_lost_by_inner': 25, 'duplicate_lookup_join_rows': 6855, 'rows_added_by_duplicate_key': 25, 'expected_validation': 'many_to_one', 'note': 'Учебная неполная копия справочника: удалена первая строка lookup.'}


## 05. Профиль с собственным знаменателем

In [13]:
prepared = decisions.copy()
prepared["is_approved"] = prepared["action_taken"].isin([1,2])
profile = prepared.groupby(org_keys).agg(n_decisions=("action_taken","size"), n_approved=("is_approved","sum")).reset_index()
profile["approval_rate"] = profile["n_approved"] / profile["n_decisions"]
named_profile = profile.merge(lookup, on=org_keys, how="left", validate="one_to_one", indicator=True)
assert named_profile["_merge"].eq("both").all()
named_profile = named_profile.drop(columns="_merge")
assert len(named_profile) == 314
assert named_profile["n_decisions"].sum() == 6830
assert named_profile["n_approved"].sum() == 6139
named_profile.sort_values("n_decisions", ascending=False).head(5)

                      lei  ...                   organization_name
119  549300CF8MP6S7MZV277  ...  FIRST SAVINGS MORTGAGE CORPORATION
147  549300GPO6DWUZR4UY30  ...     FIRST HOME MORTGAGE CORPORATION
290  E57ODZWZ7FF32TWEFA76  ...      Citibank, National Association
81   5493005PKOSG7MYX0B34  ...       Prosperity Home Mortgage, LLC
296  JJKC32MCHWDI71265Z06  ...                         Truist Bank

[5 rows x 6 columns]

## E. Передайте проверяемый результат

In [14]:
named_profile.to_csv(WORK / "organization_profile.csv", index=False)
assert named_profile["approval_rate"].between(0, 1).all()
assert named_profile["n_decisions"].gt(0).all()
assert named_profile["n_approved"].le(named_profile["n_decisions"]).all()
print("Сохранён:", WORK / "organization_profile.csv")
print("Число организаций:", len(named_profile))
print("Общий approval rate:", named_profile["n_approved"].sum() / named_profile["n_decisions"].sum())
print("Простое среднее ставок организаций:", named_profile["approval_rate"].mean())
# Эти две величины различаются: первая взвешена числом решений, вторая даёт каждой организации одинаковый вес.
profile_check = pd.read_csv(WORK / "organization_profile.csv", dtype={"lei": "string"}, float_precision="round_trip")
assert_frame_equal(named_profile, profile_check, check_exact=True)
print("Повторное чтение CSV: профиль совпадает.")

Сохранён: ../results/organization_profile.csv
Число организаций: 314
Общий approval rate: 0.8988286969253294
Простое среднее ставок организаций: 0.8577762686344285
Повторное чтение CSV: профиль совпадает.
